In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

In [2]:
countries = pd.read_csv(RAW / "countries.csv")
corruption = pd.read_csv(RAW / "corruption_index.csv")
fatf = pd.read_csv(RAW / "fatf_high_risk.csv")
income = pd.read_csv(RAW / "income_levels.csv")

In [3]:
countries.columns = countries.columns.str.lower().str.replace(" ", "_")
corruption.columns = corruption.columns.str.lower().str.replace(" ", "_")
fatf.columns = fatf.columns.str.lower().str.replace(" ", "_")
income.columns = income.columns.str.lower().str.replace(" ", "_")

In [4]:
countries["country_code"] = countries["country_code"].str.upper()

In [5]:
dim_countries = countries.copy()

In [6]:
dim_countries = dim_countries.merge(
    corruption,
    on="country_code",
    how="left"
)

In [7]:
dim_countries = dim_countries.merge(
    income,
    on="country_code",
    how="left"
)

In [8]:
dim_countries = dim_countries.merge(
    countries[["country_code", "country_name"]],
    on="country_code",
    how="left"
)

In [9]:
dim_countries["fatf_risk"] = dim_countries["country_code"].isin(
    fatf["country_code"]
).map({True: "High", False: "Low"})

In [10]:
dim_countries.rename(columns={
    "corruption_index": "corruption_score"
}, inplace=True)

In [11]:
dim_countries = dim_countries[
    [
        "country_name",
        "country_code",
        "continent",
        "income_level",
        "cpi_score",
        "fatf_risk"
    ]
]

In [12]:
dim_countries = dim_countries.fillna("Unidentified")

In [13]:
dim_countries.isnull().sum()

country_name    0
country_code    0
continent       0
income_level    0
cpi_score       0
fatf_risk       0
dtype: int64

In [14]:
dim_countries.duplicated().sum()

np.int64(0)

In [15]:
dim_countries.head()

,country_name,country_code,continent,income_level,cpi_score,fatf_risk
0,Afghanistan,AF,Asia,Low income,16.0,Low
1,Åland Islands,AX,Europe,Unidentified,Unidentified,Low
2,Albania,AL,Europe,Upper middle income,39.0,Low
3,Algeria,DZ,Africa,Upper middle income,34.0,High
4,American Samoa,AS,Oceania,Unidentified,Unidentified,Low


In [16]:
OUTPUT = Path("../data/processed")

dim_countries.to_csv(
    OUTPUT / "dim_countries.csv",
    index=False
)